# PV Yield Forecaster — reproducible analysis

This notebook reproduces the report tables and figures using NASA POWER data.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'src'))
from pvyield.cities import CITIES
from pvyield.data import fetch_daily
from pvyield.evaluate import compare, feature_importance
from pvyield.ml import train_forecaster
from pvyield.plots import daily_yield_figure, feature_importance_figure
from IPython.display import display


In [ ]:
start, end, system_kw = '2019-01-01', '2024-12-31', 5.0
results = {city: compare(city, start, end, system_kw) for city in ['Faisalabad', 'Karachi', 'Quetta', 'Skardu']}
display(pd.concat(results, names=['city', 'model']))

In [ ]:
weather = fetch_daily('Faisalabad', start, end)
trained = train_forecaster(weather, CITIES['Faisalabad']['lat'], system_kw=system_kw)
test = trained['split'].test
actual = test['target']
predicted = pd.Series(trained['model'].predict(test.drop(columns=['target'])), index=test.index)
daily_yield_figure(pd.concat([actual.rename('actual'), predicted.rename('predicted')], axis=1)).show()
feature_importance_figure(feature_importance(trained)).show()